# 12 - Visualizaciones para el TFM

Genera las figuras y tablas agregadas para la memoria del TFM.

Pensado para GitHub y trabajo en equipo:
- No usa rutas absolutas.
- Busca automaticamente la raiz del repositorio.
- Lee solo los CSV finales versionados en `data/raw/`.
- Exporta resultados agregados a `docs/overleaf/figures/` y `docs/overleaf/tables/`.
- No exporta texto original de tweets, usernames ni identificadores personales.

Estilo cientifico: las figuras se generan SIN titulo embebido. El titulo
se anade en LaTeX mediante `\caption{}`, siguiendo el estandar APA 7.

Para ejecutarlo en local:
```bash
pip install -r requirements.txt
jupyter notebook
```


## 0. Configuracion reproducible

No es una visualizacion. Prepara rutas, estilo y funciones auxiliares.

Las figuras se guardan en PNG (borrador) y PDF (Overleaf, maxima calidad).


In [ ]:
from pathlib import Path
import warnings

try:
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    from matplotlib.ticker import FuncFormatter, MaxNLocator
    from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        f"Falta la libreria '{exc.name}'. Instala las dependencias desde la raiz del repositorio con: "
        "pip install -r requirements.txt"
    ) from exc

warnings.filterwarnings("ignore", category=FutureWarning)

def find_repo_root(start=None):
    start = Path.cwd().resolve() if start is None else Path(start).resolve()
    for candidate in [start, *start.parents]:
        has_data = (candidate / "data" / "raw").exists()
        has_overleaf = (candidate / "docs" / "overleaf").exists()
        if has_data and has_overleaf:
            return candidate
    raise FileNotFoundError(
        "No encuentro la raiz del repositorio. Ejecuta desde el repo o desde la carpeta notebooks/."
    )

REPO_ROOT = find_repo_root()
DATA_RAW = REPO_ROOT / "data" / "raw"
OVERLEAF = REPO_ROOT / "docs" / "overleaf"
FIG_DIR = OVERLEAF / "figures"
TABLE_DIR = OVERLEAF / "tables"

FIG_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

US_CSV = DATA_RAW / "scam_us_FINAL_classified.csv"
ES_CSV = DATA_RAW / "scam_es_FINAL_classified_corregido.csv"

for path in [US_CSV, ES_CSV]:
    if not path.exists():
        raise FileNotFoundError(f"No encuentro {path.relative_to(REPO_ROOT)}")

plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "font.size": 10,
    "axes.titlesize": 13,
    "axes.labelsize": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.22,
    "grid.linewidth": 0.7,
    "legend.frameon": False,
})

COUNTRY_COLORS = {
    "Estados Unidos": "#2F6B8F",
    "Espana": "#C75646",
}

CATEGORY_LABELS = {
    "phishing_identity": "Phishing / robo de identidad",
    "investment_crypto": "Inversion / cripto",
    "ponzi_pyramid": "Ponzi / piramidal",
    "employment": "Empleo",
    "romance": "Romance",
    "bank_wire": "Banco / transferencias",
    "gov_impersonation": "Suplantacion publica",
    "tax": "Fiscal",
    "tech_support": "Soporte tecnico",
    "payment_app": "Apps de pago",
    "charity": "Caridad / donaciones",
    "corporate": "Corporativo / bursatil",
    "insurance": "Seguros",
    "not_related": "No relacionado",
}

TOPIC_STOPWORDS = {
    "a", "an", "and", "are", "as", "be", "by", "for", "from", "in", "is", "it", "of", "on", "or",
    "that", "the", "this", "to", "will", "with", "your", "you", "de", "del", "la", "el", "en", "que",
    "por", "una", "un", "los", "las", "para", "con", "se", "su", "sus"
}

def pct_formatter(x, pos):
    return f"{x:.0f}%"

def label_category(code):
    return CATEGORY_LABELS.get(code, code)

def save_figure(fig, name):
    """Guarda una figura en PNG y PDF."""
    png_path = FIG_DIR / f"{name}.png"
    pdf_path = FIG_DIR / f"{name}.pdf"
    fig.savefig(png_path, bbox_inches="tight")
    fig.savefig(pdf_path, bbox_inches="tight")
    print(f"Guardado: {png_path.relative_to(REPO_ROOT)}")
    print(f"Guardado: {pdf_path.relative_to(REPO_ROOT)}")
    return png_path, pdf_path

def export_table(df, name, caption=None, label=None):
    """Exporta una tabla agregada a CSV y a LaTeX."""
    csv_path = TABLE_DIR / f"{name}.csv"
    tex_path = TABLE_DIR / f"{name}.tex"
    df.to_csv(csv_path, index=False, encoding="utf-8")
    print(f"Guardado: {csv_path.relative_to(REPO_ROOT)}")
    try:
        df.to_latex(tex_path, index=False, escape=True, caption=caption, label=label)
        print(f"Guardado: {tex_path.relative_to(REPO_ROOT)}")
    except Exception as exc:
        print(f"No se pudo exportar LaTeX para {name}: {exc}")
    return csv_path

print(f"Raiz del repo: {REPO_ROOT}")
print(f"Figuras: {FIG_DIR.relative_to(REPO_ROOT)}")
print(f"Tablas: {TABLE_DIR.relative_to(REPO_ROOT)}")


## 1. Carga de datos finales

Carga los corpus finales que sostienen todas las figuras.

Entradas:
- `data/raw/scam_us_FINAL_classified.csv`
- `data/raw/scam_es_FINAL_classified_corregido.csv`


In [ ]:
us = pd.read_csv(US_CSV)
es = pd.read_csv(ES_CSV)

for df, country in [(us, "Estados Unidos"), (es, "Espana")]:
    df["country"] = country
    df["created_at"] = pd.to_datetime(df["created_at"], errors="coerce", utc=True)
    df["month"] = df["created_at"].dt.to_period("M").astype(str)
    df["confidence_score"] = pd.to_numeric(df["confidence_score"], errors="coerce")
    for col in ["retweet_count", "reply_count", "like_count", "quote_count", "n_words", "n_hashtags", "n_mentions", "n_urls", "user_followers"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

combined = pd.concat([us, es], ignore_index=True)

print(f"EEUU:   {len(us):,} tweets")
print(f"Espana: {len(es):,} tweets")


## 2. Tabla descriptiva de los corpus

**Donde:** Materiales y Metodos, despues de Estructura de los Datos.


In [ ]:
def corpus_summary(df, country):
    relevant = df["is_relevant"].astype(str).str.lower().eq("true") if "is_relevant" in df.columns else pd.Series([np.nan] * len(df))
    return {
        "Corpus": country,
        "Tweets": len(df),
        "Periodo": f"{df['created_at'].min().date()} - {df['created_at'].max().date()}",
        "Relevantes (%)": round(relevant.mean() * 100, 1),
        "Palabras medias": round(df["n_words"].mean(), 1) if "n_words" in df.columns else None,
        "URLs medias": round(df["n_urls"].mean(), 2) if "n_urls" in df.columns else None,
        "Hashtags medios": round(df["n_hashtags"].mean(), 2) if "n_hashtags" in df.columns else None,
        "Likes mediana": round(df["like_count"].median(), 1) if "like_count" in df.columns else None,
        "Retweets mediana": round(df["retweet_count"].median(), 1) if "retweet_count" in df.columns else None,
        "Confianza media": round(df["confidence_score"].mean(), 3),
    }

tabla_corpus = pd.DataFrame([
    corpus_summary(us, "Estados Unidos"),
    corpus_summary(es, "Espana"),
])

export_table(
    tabla_corpus,
    "tabla_caracteristicas_corpus",
    caption="Caracteristicas descriptivas de los corpus finales.",
    label="tab:caracteristicas_corpus",
)

display(tabla_corpus)


## 3a. Figura: pipeline metodologico (version con iconos)

**Donde:** Materiales y Metodos, al inicio del capitulo o en Diseno general.

**Titulo sugerido:** `Pipeline metodologico del estudio`.

**Leyenda sugerida:** `Etapas del flujo de trabajo: extraccion, preprocesamiento, exploracion tematica, clasificacion, post-correccion y analisis comparativo. Las cuatro primeras se aplican a los dos corpus; la post-correccion solo al espanol.`


In [ ]:
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch, Circle, Polygon, Rectangle, Wedge

stages = [
    {"num": "1", "title": "Extraccion",         "desc": "API de X (Twitter)\nbusqueda por\npalabras clave",
     "color": "#5B8BA8", "icon": "download"},
    {"num": "2", "title": "Preproceso",         "desc": "Limpieza textual\nfiltros anti-bot\ny geograficos",
     "color": "#5B8BA8", "icon": "filter"},
    {"num": "3", "title": "BERTopic",           "desc": "Exploracion\ntematica\nno supervisada",
     "color": "#6C8E4E", "icon": "cluster"},
    {"num": "4", "title": "BART-MNLI",          "desc": "Clasificacion\nzero-shot\n14 categorias",
     "color": "#6C8E4E", "icon": "tags"},
    {"num": "5", "title": "Post-correccion",    "desc": "Regla objetiva\nsobre el hijo\nen apuros",
     "color": "#C75646", "icon": "wrench"},
    {"num": "6", "title": "Analisis",           "desc": "Distribuciones,\ndivergencias y\ntransferibilidad",
     "color": "#4C7C7D", "icon": "chart"},
]

def draw_icon(ax, kind, cx, cy, size, color):
    s = size
    if kind == "download":
        ax.add_patch(Polygon([[cx-s*0.35, cy+s*0.1], [cx-s*0.12, cy+s*0.1],
                              [cx-s*0.12, cy+s*0.45], [cx+s*0.12, cy+s*0.45],
                              [cx+s*0.12, cy+s*0.1], [cx+s*0.35, cy+s*0.1],
                              [cx, cy-s*0.35]],
                             closed=True, facecolor=color, edgecolor=color))
        ax.add_patch(Rectangle((cx-s*0.45, cy-s*0.55), s*0.9, s*0.12,
                               facecolor=color, edgecolor=color))
    elif kind == "filter":
        ax.add_patch(Polygon([[cx-s*0.45, cy+s*0.4], [cx+s*0.45, cy+s*0.4],
                              [cx+s*0.12, cy-s*0.05], [cx+s*0.12, cy-s*0.45],
                              [cx-s*0.12, cy-s*0.45], [cx-s*0.12, cy-s*0.05]],
                             closed=True, facecolor="none", edgecolor=color, linewidth=2.2))
    elif kind == "cluster":
        pts = [(-0.30,0.20),(-0.10,0.35),(0.15,0.25),(0.30,0.05),
               (-0.25,-0.10),(0.05,-0.05),(-0.05,-0.30),(0.25,-0.25)]
        for dx,dy in pts:
            ax.add_patch(Circle((cx+dx*s, cy+dy*s), s*0.07, facecolor=color, edgecolor=color))
    elif kind == "tags":
        for k, dy in enumerate([0.28, 0.0, -0.28]):
            ax.add_patch(Polygon([[cx-s*0.40, cy+dy*s+s*0.08], [cx+s*0.25, cy+dy*s+s*0.08],
                                  [cx+s*0.40, cy+dy*s], [cx+s*0.25, cy+dy*s-s*0.08],
                                  [cx-s*0.40, cy+dy*s-s*0.08]],
                                 closed=True, facecolor="none", edgecolor=color, linewidth=1.8))
            ax.add_patch(Circle((cx-s*0.30, cy+dy*s), s*0.04, facecolor=color, edgecolor=color))
    elif kind == "wrench":
        ax.plot([cx-s*0.30, cx+s*0.30], [cy-s*0.30, cy+s*0.30], color=color, linewidth=3.5)
        ax.add_patch(Wedge((cx-s*0.30, cy-s*0.30), s*0.20, 30, 320,
                           facecolor="none", edgecolor=color, linewidth=2.8))
    elif kind == "chart":
        for k, h in enumerate([0.30, 0.55, 0.40, 0.65]):
            x = cx + (k-1.5)*s*0.18
            ax.add_patch(Rectangle((x-s*0.07, cy-s*0.40), s*0.14, s*h,
                                   facecolor=color, edgecolor=color, alpha=0.85))

# Figura mas alta y mas ancha para que respire
fig, ax = plt.subplots(figsize=(16, 7.5))
ax.set_axis_off()
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)

n = len(stages)
# CAJAS MAS GRANDES Y MAS ESPACIADAS
box_w = 0.155   # antes 0.135
box_h = 0.66    # antes 0.50 -- MUCHO mas alta
y_center = 0.50
x_positions = np.linspace(0.090, 0.910, n)

for i, (st, x) in enumerate(zip(stages, x_positions)):
    color = st["color"]
    patch = FancyBboxPatch(
        (x - box_w/2, y_center - box_h/2),
        box_w, box_h,
        boxstyle="round,pad=0.012,rounding_size=0.022",
        linewidth=2.0, edgecolor=color, facecolor="#FBFCFD",
        transform=ax.transAxes,
    )
    ax.add_patch(patch)

    # numero esquina superior izquierda
    ax.text(x - box_w/2 + 0.014, y_center + box_h/2 - 0.040, st["num"],
            ha="left", va="top", fontsize=10, color=color, fontweight="bold",
            transform=ax.transAxes)

    # icono cerca de la parte superior
    draw_icon(ax, st["icon"], x, y_center + 0.16, 0.085, color)

    # titulo en el centro-arriba
    ax.text(x, y_center + 0.01, st["title"],
            ha="center", va="center", fontsize=12, color="#1F2933",
            fontweight="bold", transform=ax.transAxes)

    # descripcion DEBAJO del titulo, con espacio
    ax.text(x, y_center - 0.16, st["desc"],
            ha="center", va="center", fontsize=8.8, color="#4F5B66",
            transform=ax.transAxes, linespacing=1.5)

    if i < n - 1:
        arrow = FancyArrowPatch(
            (x + box_w/2 + 0.006, y_center),
            (x_positions[i+1] - box_w/2 - 0.006, y_center),
            arrowstyle="-|>", mutation_scale=18,
            linewidth=1.8, color="#9AA0A6",
            transform=ax.transAxes,
        )
        ax.add_patch(arrow)

# anotaciones DEBAJO con MAS espacio
ax.annotate(
    "Aplicado en paralelo a corpus EEUU y corpus Espana",
    xy=(0.30, y_center - box_h/2 - 0.06), xycoords="axes fraction",
    ha="center", va="top", fontsize=10, color="#5B8BA8", style="italic",
)
ax.annotate(
    "Solo corpus espanol",
    xy=(x_positions[4], y_center - box_h/2 - 0.06), xycoords="axes fraction",
    ha="center", va="top", fontsize=10, color="#C75646", style="italic",
)

save_figure(fig, "fig_00a_pipeline_color")
plt.show()


## 3b. Figura: pipeline metodologico (version blanco y negro)

Version alternativa, sobria, estilo paper cientifico. Util si quereis usar una version en cuerpo y otra en anexo, o si preferis tono mas formal.


In [ ]:
stages_bn = [
    ("1", "Extraccion",           "API de X"),
    ("2", "Preproceso",           "Limpieza textual"),
    ("3", "BERTopic",             "Topic modeling"),
    ("4", "BART-MNLI",            "Clasificacion zero-shot"),
    ("5", "Post-correccion",      "Regla objetiva (ES)"),
    ("6", "Analisis comparativo", "Resultados"),
]

fig, ax = plt.subplots(figsize=(15, 4.5))
ax.set_axis_off()
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)

n = len(stages_bn)
# CAJAS MAS GRANDES
box_w = 0.15   # antes 0.13
box_h = 0.58   # antes 0.42
y_center = 0.50
x_positions = np.linspace(0.090, 0.910, n)

EDGE = "#222222"
TEXT = "#222222"
SUBTEXT = "#555555"

for i, ((num, title, sub), x) in enumerate(zip(stages_bn, x_positions)):
    patch = FancyBboxPatch(
        (x - box_w/2, y_center - box_h/2),
        box_w, box_h,
        boxstyle="round,pad=0.010,rounding_size=0.018",
        linewidth=1.3, edgecolor=EDGE, facecolor="white",
        transform=ax.transAxes,
    )
    ax.add_patch(patch)

    # numero arriba (en circulo)
    ax.text(x, y_center + box_h/2 - 0.10, num,
            ha="center", va="center", fontsize=12, color=TEXT,
            fontweight="bold", transform=ax.transAxes,
            bbox=dict(boxstyle="circle,pad=0.22", facecolor="white",
                      edgecolor=EDGE, linewidth=1.0))

    # titulo
    ax.text(x, y_center - 0.02, title,
            ha="center", va="center", fontsize=11, color=TEXT,
            fontweight="bold", transform=ax.transAxes)

    # subtitulo / descripcion
    ax.text(x, y_center - 0.16, sub,
            ha="center", va="center", fontsize=9, color=SUBTEXT,
            style="italic", transform=ax.transAxes)

    if i < n - 1:
        arrow = FancyArrowPatch(
            (x + box_w/2 + 0.005, y_center),
            (x_positions[i+1] - box_w/2 - 0.005, y_center),
            arrowstyle="-|>", mutation_scale=15,
            linewidth=1.3, color="#444444",
            transform=ax.transAxes,
        )
        ax.add_patch(arrow)

save_figure(fig, "fig_00b_pipeline_bn")
plt.show()


## 4. Figura: flujo de construccion del corpus

**Donde:** Materiales y Metodos, en Ingesta de datos o antes de Preprocesamiento.

**Titulo sugerido:** `Flujo de construccion del corpus de analisis`.

**Leyenda sugerida:** `Volumen de tweets en cada etapa del pipeline para los corpus estadounidense y espanol. Se muestran las etapas con cifra verificable; los CSV intermedios del corpus espanol no estan versionados.`


In [ ]:
pipeline_us = [
    ("Concatenado tras\ndeduplicacion", 12726),
    ("Filtros\nestructurales", 2324),
    ("Limpieza final\ny cap usuario", len(us)),
    ("Clasificados\nrelevantes", int(us["is_relevant"].astype(str).str.lower().eq("true").sum())),
]

pipeline_es = [
    ("Corpus final\nEspana", len(es)),
    ("Clasificados\nrelevantes", int(es["is_relevant"].astype(str).str.lower().eq("true").sum())),
]

def draw_pipeline(ax, steps, y, color, label):
    ax.text(0.02, y + 0.14, label, fontsize=12, fontweight="bold", color=color, transform=ax.transAxes)
    x_positions = np.linspace(0.12, 0.88, len(steps))
    box_w = min(0.18, 0.74 / max(len(steps), 1))
    box_h = 0.16
    for i, ((name, value), x) in enumerate(zip(steps, x_positions)):
        patch = FancyBboxPatch(
            (x - box_w / 2, y - box_h / 2), box_w, box_h,
            boxstyle="round,pad=0.018,rounding_size=0.018",
            linewidth=1.2, edgecolor=color, facecolor="#F7F9FB", transform=ax.transAxes,
        )
        ax.add_patch(patch)
        ax.text(x, y + 0.02, name, ha="center", va="center", fontsize=9, transform=ax.transAxes)
        ax.text(x, y - 0.05, f"{value:,}".replace(",", "."), ha="center", va="center", fontsize=11, fontweight="bold", color=color, transform=ax.transAxes)
        if i < len(steps) - 1:
            arrow = FancyArrowPatch(
                (x + box_w / 2 + 0.01, y), (x_positions[i + 1] - box_w / 2 - 0.01, y),
                arrowstyle="-|>", mutation_scale=14, linewidth=1.2, color="#6E7781", transform=ax.transAxes,
            )
            ax.add_patch(arrow)

fig, ax = plt.subplots(figsize=(12, 4.3))
ax.set_axis_off()
draw_pipeline(ax, pipeline_us, y=0.68, color=COUNTRY_COLORS["Estados Unidos"], label="Estados Unidos")
draw_pipeline(ax, pipeline_es, y=0.28, color=COUNTRY_COLORS["Espana"], label="Espana")
save_figure(fig, "fig_01_flujo_construccion_corpus")
plt.show()


## 5. Figura y tabla: distribucion comparada de categorias

**Donde:** Resultados, primera visualizacion principal.

**Titulo sugerido:** `Distribucion comparada de tipologias de fraude en EE. UU. y Espana`.

**Leyenda sugerida:** `Porcentaje de tweets asignados a cada categoria por el clasificador zero-shot BART-MNLI. En Espana se emplea la categoria corregida tras la regla de post-correccion sobre la estafa del hijo en apuros.`


In [ ]:
def category_distribution(df, prefix):
    counts = df["predicted_category"].value_counts()
    out = counts.rename(f"{prefix}_n").to_frame()
    out[f"{prefix}_pct"] = out[f"{prefix}_n"] / len(df) * 100
    return out

comp = category_distribution(us, "us").join(category_distribution(es, "es"), how="outer").fillna(0)
comp = comp.reset_index(names="category")
comp["category_label"] = comp["category"].map(label_category)
comp["delta_es_us_pp"] = comp["es_pct"] - comp["us_pct"]
comp = comp.sort_values(["us_pct", "es_pct"], ascending=False).reset_index(drop=True)

tabla_categorias = comp[["category_label", "us_n", "us_pct", "es_n", "es_pct", "delta_es_us_pp"]].copy()
tabla_categorias.columns = ["Categoria", "EEUU n", "EEUU %", "Espana n", "Espana %", "Diferencia Espana-EEUU (p.p.)"]
for col in ["EEUU %", "Espana %", "Diferencia Espana-EEUU (p.p.)"]:
    tabla_categorias[col] = tabla_categorias[col].round(1)

export_table(
    tabla_categorias,
    "tabla_distribucion_categorias",
    caption="Frecuencia absoluta y relativa por categoria predicha.",
    label="tab:distribucion_categorias",
)

plot_df = comp.sort_values("us_pct", ascending=True)
y = np.arange(len(plot_df))
height = 0.38

fig, ax = plt.subplots(figsize=(10.5, 7.5))
bars_us = ax.barh(y - height / 2, plot_df["us_pct"], height, label="Estados Unidos", color=COUNTRY_COLORS["Estados Unidos"])
bars_es = ax.barh(y + height / 2, plot_df["es_pct"], height, label="Espana", color=COUNTRY_COLORS["Espana"])
ax.set_yticks(y)
ax.set_yticklabels(plot_df["category_label"])
ax.xaxis.set_major_formatter(FuncFormatter(pct_formatter))
ax.set_xlabel("Porcentaje del corpus")
ax.legend(loc="lower right")
ax.set_xlim(0, max(plot_df["us_pct"].max(), plot_df["es_pct"].max()) * 1.18)
ax.bar_label(bars_us, labels=[f"{v:.1f}%" if v >= 2 else "" for v in plot_df["us_pct"]], padding=3, fontsize=8)
ax.bar_label(bars_es, labels=[f"{v:.1f}%" if v >= 2 else "" for v in plot_df["es_pct"]], padding=3, fontsize=8)
fig.tight_layout()
save_figure(fig, "fig_02_distribucion_categorias_comparada")
plt.show()


## 6. Figura: evolucion mensual de las categorias principales

**Donde:** Resultados, despues de la distribucion general.

**Titulo sugerido:** `Evolucion temporal de las principales tipologias de fraude durante 2025`.

**Leyenda sugerida:** `Frecuencia mensual de las cinco categorias mas representadas en cada corpus. Cada panel emplea las categorias principales de su propio contexto nacional.`


In [ ]:
CATEGORY_COLORS = [
    "#2F6B8F", "#C75646", "#6C8E4E", "#9A6FB0", "#D99C2B", "#4C7C7D", "#8C564B", "#7F7F7F"
]

def monthly_top_categories(df, top_n=5):
    top = df["predicted_category"].value_counts().head(top_n).index.tolist()
    months = pd.period_range("2025-01", "2025-12", freq="M").astype(str)
    monthly = (
        df[df["predicted_category"].isin(top)]
        .groupby(["month", "predicted_category"])
        .size()
        .unstack(fill_value=0)
        .reindex(months, fill_value=0)
    )
    return monthly[top]

monthly_us = monthly_top_categories(us)
monthly_es = monthly_top_categories(es)

fig, axes = plt.subplots(2, 1, figsize=(11, 8), sharex=True)
for ax, monthly, panel_label in [
    (axes[0], monthly_us, "Estados Unidos"),
    (axes[1], monthly_es, "Espana"),
]:
    for i, category in enumerate(monthly.columns):
        ax.plot(monthly.index, monthly[category], marker="o", linewidth=2, markersize=4, label=label_category(category), color=CATEGORY_COLORS[i])
    # Etiqueta del panel en la esquina superior izquierda, sin ser titulo
    ax.text(0.01, 0.96, panel_label, transform=ax.transAxes,
            fontsize=11, fontweight="bold", va="top",
            color=COUNTRY_COLORS[panel_label])
    ax.set_ylabel("Tweets")
    ax.yaxis.set_major_locator(MaxNLocator(integer=True))
    ax.legend(ncol=2, fontsize=8, loc="upper right")

axes[-1].set_xlabel("Mes de publicacion")
axes[-1].tick_params(axis="x", rotation=45)
fig.tight_layout()
save_figure(fig, "fig_03_evolucion_mensual_categorias")
plt.show()


## 7. Figura: post-correccion espanola de la estafa del hijo en apuros

**Donde:** Materiales y Metodos (decision metodologica) o Resultados (diagnostico).

**Titulo sugerido:** `Efecto de la post-correccion de la estafa del hijo en apuros`.

**Leyenda sugerida:** `Comparacion de frecuencias antes y despues de aplicar la regla objetiva que reasigna los tweets de la estafa del hijo en apuros desde fraude de seguros hacia phishing o robo de identidad en el corpus espanol.`


In [ ]:
if "predicted_category_original" not in es.columns:
    print("El CSV espanol no contiene predicted_category_original; se omite esta figura.")
else:
    before = es["predicted_category_original"].fillna(es["predicted_category"])
    after = es["predicted_category"]
    changed = before.ne(after)
    corrected_n = int(changed.sum())
    focus_categories = ["phishing_identity", "insurance"]
    corr_df = pd.DataFrame({
        "Categoria": [label_category(cat) for cat in focus_categories],
        "Antes": [int((before == cat).sum()) for cat in focus_categories],
        "Despues": [int((after == cat).sum()) for cat in focus_categories],
    })
    export_table(
        corr_df,
        "tabla_postcorreccion_hijo_apuros",
        caption="Efecto de la post-correccion de la estafa del hijo en apuros.",
        label="tab:postcorreccion_hijo_apuros",
    )

    x = np.arange(len(corr_df))
    width = 0.36
    fig, ax = plt.subplots(figsize=(9.5, 5.4))
    bars_before = ax.bar(x - width / 2, corr_df["Antes"], width, label="Antes", color="#8A8F98")
    bars_after = ax.bar(x + width / 2, corr_df["Despues"], width, label="Despues", color=COUNTRY_COLORS["Espana"])
    ax.set_xticks(x)
    ax.set_xticklabels(corr_df["Categoria"])
    ax.set_ylabel("Tweets")
    ax.set_ylim(0, max(corr_df[["Antes", "Despues"]].to_numpy().max() * 1.18, 50))
    ax.legend()
    ax.bar_label(bars_before, padding=3)
    ax.bar_label(bars_after, padding=3)
    ax.text(
        0.02, 0.90, f"Tweets reasignados: {corrected_n}",
        transform=ax.transAxes, fontsize=10.5, fontweight="bold", color=COUNTRY_COLORS["Espana"],
        bbox={"boxstyle": "round,pad=0.35", "facecolor": "white", "edgecolor": COUNTRY_COLORS["Espana"], "linewidth": 1.0, "alpha": 0.96},
        zorder=10, clip_on=False,
    )
    fig.tight_layout()
    save_figure(fig, "fig_06_postcorreccion_hijo_apuros")
    plt.show()
    display(corr_df)


## 8. Figura: topicos BERTopic del corpus estadounidense

**Donde:** Resultados, subseccion de exploracion tematica.

**Titulo sugerido:** `Principales topicos emergentes en el corpus estadounidense`.

**Leyenda sugerida:** `Topicos generados mediante BERTopic sobre el corpus estadounidense. Se excluye el topico -1 (outliers). Las etiquetas muestran palabras clave representativas tras retirar conectores frecuentes.`


In [ ]:
def clean_topic_label(keyword_string, max_words=4):
    words = []
    for raw in str(keyword_string).split(","):
        word = raw.strip().lower()
        if word and word not in TOPIC_STOPWORDS and word not in words:
            words.append(word)
    return ", ".join(words[:max_words]) if words else str(keyword_string)

def topic_summary(df, top_n=10):
    if "bertopic_id" not in df.columns or "bertopic_keywords" not in df.columns:
        return pd.DataFrame()
    topics = (
        df.groupby(["bertopic_id", "bertopic_keywords"])
        .size()
        .reset_index(name="n_tweets")
    )
    topics["bertopic_id_num"] = pd.to_numeric(topics["bertopic_id"], errors="coerce")
    topics = topics[topics["bertopic_id_num"] != -1]
    topics["pct"] = topics["n_tweets"] / len(df) * 100
    topics["topic_label"] = topics["bertopic_keywords"].map(clean_topic_label)
    return topics.sort_values("n_tweets", ascending=False).head(top_n)

topics_us = topic_summary(us, top_n=10)
topics_us_table = topics_us[["bertopic_id", "topic_label", "n_tweets", "pct", "bertopic_keywords"]].copy()
topics_us_table.columns = ["Topic ID", "Etiqueta visual", "Tweets", "% corpus", "Keywords originales"]
topics_us_table["% corpus"] = topics_us_table["% corpus"].round(1)

export_table(
    topics_us_table,
    "tabla_topicos_bertopic_eeuu",
    caption="Topicos BERTopic principales del corpus estadounidense.",
    label="tab:topicos_bertopic_eeuu",
)

plot_topics = topics_us.sort_values("n_tweets", ascending=True)
fig, ax = plt.subplots(figsize=(9.5, 5.8))
bars = ax.barh(plot_topics["topic_label"], plot_topics["n_tweets"], color=COUNTRY_COLORS["Estados Unidos"])
ax.set_xlabel("Tweets")
ax.bar_label(bars, labels=[f"{n} ({p:.1f}%)" for n, p in zip(plot_topics["n_tweets"], plot_topics["pct"])], padding=3, fontsize=8)
ax.set_xlim(0, plot_topics["n_tweets"].max() * 1.22)
fig.tight_layout()
save_figure(fig, "fig_07_topicos_bertopic_eeuu")
plt.show()

display(topics_us_table)


## 9. Figura: topicos BERTopic del corpus espanol

**Donde:** Resultados, subseccion de exploracion tematica.

**Titulo sugerido:** `Principales topicos emergentes en el corpus espanol`.

**Leyenda sugerida:** `Topicos generados mediante BERTopic sobre el corpus espanol con stopwords filtradas. Se excluye el topico -1 (outliers).`


In [ ]:
topics_es = topic_summary(es, top_n=10)

if topics_es.empty:
    print("No hay columnas BERTopic en el corpus espanol.")
else:
    largest_share = topics_es["pct"].max()
    if largest_share > 70:
        print(f"AVISO: el topico principal supera el 70% del corpus ({largest_share:.1f}%).")
        print("Recomendacion: ejecutar notebooks/11_bertopic_espana_stopwords.ipynb antes de generar la figura.")
    else:
        topics_es_table = topics_es[["bertopic_id", "topic_label", "n_tweets", "pct", "bertopic_keywords"]].copy()
        topics_es_table.columns = ["Topic ID", "Etiqueta visual", "Tweets", "% corpus", "Keywords originales"]
        topics_es_table["% corpus"] = topics_es_table["% corpus"].round(1)

        export_table(
            topics_es_table,
            "tabla_topicos_bertopic_espana",
            caption="Topicos BERTopic principales del corpus espanol.",
            label="tab:topicos_bertopic_espana",
        )

        plot_topics = topics_es.sort_values("n_tweets", ascending=True)
        fig, ax = plt.subplots(figsize=(9.5, 5.8))
        bars = ax.barh(plot_topics["topic_label"], plot_topics["n_tweets"], color=COUNTRY_COLORS["Espana"])
        ax.set_xlabel("Tweets")
        ax.bar_label(bars, labels=[f"{n} ({p:.1f}%)" for n, p in zip(plot_topics["n_tweets"], plot_topics["pct"])], padding=3, fontsize=8)
        ax.set_xlim(0, plot_topics["n_tweets"].max() * 1.22)
        fig.tight_layout()
        save_figure(fig, "fig_08_topicos_bertopic_espana")
        plt.show()

        display(topics_es_table)


## 10. Resumen de archivos generados

Verificacion de que todo se ha exportado correctamente.

En Overleaf, las figuras se pueden incluir desde `figures/nombre_figura.pdf` o `.png`.


In [ ]:
generated_figures = sorted([p.relative_to(REPO_ROOT) for p in FIG_DIR.glob("fig_*.png")])
generated_tables = sorted([p.relative_to(REPO_ROOT) for p in TABLE_DIR.glob("*.csv")])

print("Figuras PNG generadas:")
for path in generated_figures:
    print(f"- {path}")

print("\nTablas CSV generadas:")
for path in generated_tables:
    print(f"- {path}")
